# Confidence intervals — TES (Al), heavy mediator (F_DM = 1), m_DM = 1 GeV

Point-wise profile confidence bands of the recovered flux x(v_min), one
subsection per halo model. Method and conventions:
[`quantum_sensor.statistics.find_confidence_band`](../src/quantum_sensor/statistics.py)
(neutrinoAnalysis construction: observed profile Δχ² vs an MC-calibrated
cutoff; Poisson pseudo-experiments; CLARABEL solver, thread-parallel).

- Configuration: `material='Al', q='0', mass='3', nbins=5`, background `none`
- Outputs per model: `results/TES/bkg-none/<config>/` —
  `band/band_idx*.json` (primary, one per point), `flux_profile_band.csv`
  (summary), `flux_profile_band.pdf` (figure)
- **Each `run_model` cell takes ~30–60 min** (production MC settings).
  Already computed? Use `show_saved` in §Saved results instead.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

from quantum_sensor import (DarkMatterQuantumAnalysis, RunConfig,
                            pointwise_band, load_pointwise_band,
                            band_table, save_band_products)
from quantum_sensor.plotting import plot_flux_with_pointwise_bands, run_dir

MATERIAL, Q, MASS, NBINS = 'Al', '0', '3', 5
LEVELS = (0.68, 0.954)                                   # 1 sigma, 2 sigma
MC = dict(num_pseudo=50, n_pseudo_edge=500, rel_tol=0.02)  # production settings
N_INDICES = 30


def build(eta='Halo', disk_fraction=None):
    cfg = RunConfig(material=MATERIAL, q=Q, mass=MASS, nbins=NBINS,
                    eta=eta, disk_fraction=disk_fraction, background='none')
    a = DarkMatterQuantumAnalysis(cfg)
    a.optimize()
    return a


def run_model(eta='Halo', disk_fraction=None):
    """Compute, save and display the band for one halo model (heavy)."""
    a = build(eta, disk_fraction)
    print(f'{a.config}')
    print(f'counts per bin: {np.array2string(a.observed, precision=1)}')
    bands = pointwise_band(a, n_indices=N_INDICES, levels=LEVELS, **MC)
    save_band_products(a, bands)
    display(band_table(bands))
    return a, bands


def show_saved(eta='Halo', disk_fraction=None):
    """Reload a saved band (band/*.json) — no recomputation."""
    a = build(eta, disk_fraction)
    bands = load_pointwise_band(run_dir(a))
    print(f'{len(bands)} points from {run_dir(a) / "band"}')
    display(band_table(bands))
    plot_flux_with_pointwise_bands(a, bands, save=False)
    return a, bands


## 1. Standard halo (SHM)


In [ ]:
a_halo, b_halo = run_model(eta='Halo')


RunConfig(material='Al', q='0', mass='3', nbins=5, eta='Halo', disk_fraction=None, background='none', run=None)
counts per bin: [15.8 32.4 47.6 63.8 79.8]
[1/22] idx 0 (v=5 km/s)  best 4.18e-28 cm^-1  1sigma [3.9e-28, 4.79e-27]  2sigma [3.66e-28, 9.5e-27]  (28 evals)
[2/22] idx 1 (v=5 km/s)  best 4.18e-28 cm^-1  1sigma [3.95e-28, 1.52e-27]  2sigma [3.66e-28, 3.05e-27]  (27 evals)
[3/22] idx 2 (v=6 km/s)  best 4.18e-28 cm^-1  1sigma [3.95e-28, 8.34e-28]  2sigma [3.66e-28, 1.55e-27]  (24 evals)
[4/22] idx 3 (v=7 km/s)  best 4.18e-28 cm^-1  1sigma [3.95e-28, 6.41e-28]  2sigma [3.66e-28, 1.03e-27]  (24 evals)
[5/22] idx 4 (v=8 km/s)  best 4.18e-28 cm^-1  1sigma [3.95e-28, 6.16e-28]  2sigma [3.66e-28, 8.67e-28]  (22 evals)


## 2. Pure dark disk


In [ ]:
a_disk, b_disk = run_model(eta='Disk')


## 3. Halo + dark-disk mixtures

The fit eta is `(1-p)·Halo + p·Disk` with the same total local DM density.


In [ ]:
a_mix5, b_mix5 = run_model(disk_fraction=0.05)    # 5% disk


In [ ]:
a_mix25, b_mix25 = run_model(disk_fraction=0.25)  # 25% disk


## 4. Earth-bound population (Halo + Bound)

The bound population sits on top of the full SHM halo; it is nonzero only
below v_esc = 11.2 km/s, which only the 1 GeV window reaches.


In [ ]:
a_bound, b_bound = run_model(eta='Bound')


## Saved results (no recomputation)

Reload any model computed above (or in a previous session).


In [ ]:
show_saved(eta='Halo');
# show_saved(eta='Disk');
# show_saved(disk_fraction=0.05);
# show_saved(disk_fraction=0.25);
# show_saved(eta='Bound');
